In [1]:
!curl -L -o restaurant_inspections.csv "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  138M    0  138M    0     0  1163k      0 --:--:--  0:02:01 --:--:-- 1938k


In [2]:
!ls -lh

total 139M
-rw-r--r-- 1 root root 139M Jul 25 15:39 restaurant_inspections.csv
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data


In [3]:
import pandas as pd

df = pd.read_csv("restaurant_inspections.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

Shape: (295294, 27)

Columns: ['CAMIS', 'DBA', 'BORO', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE', 'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE', 'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE', 'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude', 'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL', 'NTA', 'Location']

First few rows:


/tmp/ipykernel_4455/714086887.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("restaurant_inspections.csv")


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50177997,AWESOME HIBACHI,Queens,24-26,47 STREET,11103.0,6462098242,NaN,01/01/1900,NaN,...,NaN,40.766568,-73.905124,401.0,22.0,14100.0,4013583.0,4.007320e+09,QN70,POINT (-73.905123532108 40.766568442385)
1,50132625,THE GRIND COFFEE BAR,Staten Island,7427,AMBOY ROAD,10307.0,9179942592,Coffee/Tea,05/04/2023,Violations were cited in the following area(s).,...,Pre-permit (Operational) / Initial Inspection,40.509897,-74.244065,503.0,51.0,24800.0,5107892.0,5.080460e+09,SI11,POINT (-74.244064845648 40.509896890658)
2,50173791,EARTHBAR,Manhattan,420,LEXINGTON AVENUE,10170.0,7144251559,NaN,01/01/1900,NaN,...,NaN,40.752255,-73.975464,105.0,4.0,9200.0,1035385.0,1.012808e+09,MN19,POINT (-73.975463985528 40.752255436318)
3,50038161,SET L.E.S.,Manhattan,127,LUDLOW STREET,10002.0,2129828225,Fusion,10/29/2025,Violations were cited in the following area(s).,...,Cycle Inspection / Re-inspection,40.719790,-73.988510,103.0,1.0,1800.0,1005312.0,1.004100e+09,MN27,POINT (-73.988510004264 40.719789670042)
4,41582631,TINY'S & THE BAR UPSTAIRS,Manhattan,135,WEST BROADWAY,10013.0,2123741135,New American,05/28/2026,No violations were recorded at the time of thi...,...,Administrative Miscellaneous / Re-inspection,40.716751,-74.008232,101.0,1.0,3300.0,1001612.0,1.001470e+09,MN24,POINT (-74.00823201583 40.716751492059)


In [4]:
# Check nulls
print(df.isnull().sum())
print("\n")
# Check data types
print(df.dtypes)
print("\n")

# Check duplicates
print("Duplicate rows:", df.duplicated().sum())

# Check unique values in key categorical columns
print("\nUnique grades:", df['GRADE'].unique())
print("Unique boroughs:", df['BORO'].unique())
print("Unique critical flags:", df['CRITICAL FLAG'].unique())

CAMIS                         0
DBA                           2
BORO                          0
BUILDING                    983
STREET                       28
ZIPCODE                    3093
PHONE                        99
CUISINE DESCRIPTION        3686
INSPECTION DATE               0
ACTION                     3606
VIOLATION CODE             6285
VIOLATION DESCRIPTION      6285
CRITICAL FLAG                 0
SCORE                     17143
GRADE                    149994
GRADE DATE               160233
RECORD DATE                   0
INSPECTION TYPE            3606
Latitude                   1688
Longitude                  1688
Community Board            4744
Council District           4762
Census Tract               4762
BIN                        5970
BBL                        1688
NTA                        4744
Location                   4744
dtype: int64


CAMIS                      int64
DBA                       object
BORO                      object
BUILDING              

In [5]:
# 1. Drop exact duplicate rows
df_clean = df.drop_duplicates().copy()
print(f"Dropped {df.duplicated().sum()} duplicates, {len(df_clean)} rows remain")

# 2. Fix BORO — replace '0' with proper null/unknown marker
print("\nRows with BORO = '0':", (df_clean['BORO'] == '0').sum())
df_clean['BORO'] = df_clean['BORO'].replace('0', 'Unknown')

# 3. Standardize INSPECTION DATE to actual datetime
df_clean['INSPECTION DATE'] = pd.to_datetime(df_clean['INSPECTION DATE'], errors='coerce')
print("\nRows where date failed to parse:", df_clean['INSPECTION DATE'].isnull().sum())

# 4. Check if SCORE nulls correlate with specific inspection types
print("\nInspection types where SCORE is null:")
print(df_clean[df_clean['SCORE'].isnull()]['INSPECTION TYPE'].value_counts().head(10))

# 5. Confirm geocoding nulls correlate (Lat/Long missing = other geo fields missing)
geo_null_check = df_clean[df_clean['Latitude'].isnull()][['Community Board', 'BIN', 'NTA']].isnull().sum()
print("\nAmong rows missing Latitude, how many also missing other geo fields:")
print(geo_null_check)

# 6. Document GRADE meanings (for your own reference / README later)
grade_meanings = {
    'A': 'Grade A', 'B': 'Grade B', 'C': 'Grade C',
    'N': 'Not Yet Graded', 'Z': 'Grade Pending', 'P': 'Grade Pending (reopening)'
}
print("\nGrade distribution:")
print(df_clean['GRADE'].value_counts(dropna=False))

Dropped 139 duplicates, 295155 rows remain

Rows with BORO = '0': 381

Rows where date failed to parse: 0

Inspection types where SCORE is null:
INSPECTION TYPE
Administrative Miscellaneous / Initial Inspection       9573
Administrative Miscellaneous / Re-inspection            1786
Smoke-Free Air Act / Initial Inspection                  653
Calorie Posting / Initial Inspection                     489
Trans Fat / Initial Inspection                           452
Administrative Miscellaneous / Compliance Inspection     147
Sodium Warning / Initial Inspection                       93
Calorie Posting / Re-inspection                           80
Trans Fat / Re-inspection                                 72
Smoke-Free Air Act / Re-inspection                        72
Name: count, dtype: int64

Among rows missing Latitude, how many also missing other geo fields:
Community Board    1688
BIN                1688
NTA                1688
dtype: int64

Grade distribution:
GRADE
NaN    149884
A      

In [6]:
# 1. Drop exact duplicate rows
df_clean = df.drop_duplicates().copy()
print(f"Dropped {df.duplicated().sum()} duplicates, {len(df_clean)} rows remain")

# 2. Fix BORO — replace '0' with proper null/unknown marker
df_clean['BORO'] = df_clean['BORO'].replace('0', 'Unknown')

# 3. Standardize INSPECTION DATE to actual datetime
df_clean['INSPECTION DATE'] = pd.to_datetime(df_clean['INSPECTION DATE'], errors='coerce')

print("Clean. No warnings now.")

Dropped 139 duplicates, 295155 rows remain
Clean. No warnings now.


In [7]:
df_clean.to_csv("restaurant_inspections_clean.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (295155, 27)


In [8]:
import matplotlib.pyplot as plt

# Inspections by borough
print(df_clean['BORO'].value_counts())

# Grade distribution (excluding nulls, since those are structural)
print("\n", df_clean['GRADE'].value_counts())

# Critical flag distribution
print("\n", df_clean['CRITICAL FLAG'].value_counts())

# Top cuisine types
print("\nTop 15 cuisines:\n", df_clean['CUISINE DESCRIPTION'].value_counts().head(15))

# Score distribution (lower score = better in NYC's system)
print("\nScore stats:\n", df_clean['SCORE'].describe())

BORO
Manhattan        109308
Brooklyn          74994
Queens            73782
Bronx             27103
Staten Island      9587
Unknown             381
Name: count, dtype: int64

 GRADE
A    98149
B    18054
C    13195
N    10221
Z     4849
P      803
Name: count, dtype: int64

 CRITICAL FLAG
Critical          154951
Not Critical      132074
Not Applicable      8130
Name: count, dtype: int64

Top 15 cuisines:
 CUISINE DESCRIPTION
American                          44167
Chinese                           28974
Coffee/Tea                        21422
Pizza                             17087
Latin American                    14676
Mexican                           12119
Bakery Products/Desserts          11530
Caribbean                         11005
Japanese                          10459
Italian                            9417
Chicken                            7720
Spanish                            6153
Asian/Asian Fusion                 5742
Juice, Smoothies, Fruit Salads     5719
Sandwiche

In [9]:
df_clean['year'] = df_clean['INSPECTION DATE'].dt.year

yearly = df_clean.groupby('year').agg(
    inspection_count=('CAMIS', 'count'),
    avg_score=('SCORE', 'mean')
).reset_index()

print(yearly)

    year  inspection_count  avg_score
0   1900              3606        NaN
1   2007                51  30.468085
2   2008                29  49.714286
3   2009                 4  20.000000
4   2010                13  30.090909
5   2011                 6  16.000000
6   2012                 6  15.333333
7   2013                 2   4.000000
8   2014                12  27.000000
9   2015                33  16.500000
10  2016               249  15.174468
11  2017               465  17.796875
12  2018               698  19.932619
13  2019               818  25.010243
14  2020               217  25.376238
15  2021               344  22.869301
16  2022             15599  20.757373
17  2023             50874  24.004716
18  2024             78649  25.475727
19  2025             89573  26.230721
20  2026             53907  27.898395


In [10]:
# Only look at rows with a critical flag that's actually meaningful
critical_by_cuisine = (
    df_clean[df_clean['CRITICAL FLAG'] == 'Critical']
    .groupby('CUISINE DESCRIPTION')
    .size()
    .sort_values(ascending=False)
    .head(15)
)
print(critical_by_cuisine)

# Normalize by number of inspections per cuisine, so it's rate not raw count
total_by_cuisine = df_clean.groupby('CUISINE DESCRIPTION').size()
critical_rate = (critical_by_cuisine / total_by_cuisine).sort_values(ascending=False).dropna()
print("\nCritical violation RATE by cuisine (top 15):")
print(critical_rate.head(15))


CUISINE DESCRIPTION
American                          23003
Chinese                           16580
Coffee/Tea                        10755
Pizza                              9052
Latin American                     8058
Mexican                            6394
Bakery Products/Desserts           6164
Caribbean                          5836
Japanese                           5761
Italian                            5108
Chicken                            3889
Spanish                            3421
Asian/Asian Fusion                 3203
Juice, Smoothies, Fruit Salads     2856
Sandwiches                         2631
dtype: int64

Critical violation RATE by cuisine (top 15):
CUISINE DESCRIPTION
Chinese                           0.572237
Asian/Asian Fusion                0.557820
Spanish                           0.555989
Japanese                          0.550817
Latin American                    0.549060
Italian                           0.542423
Bakery Products/Desserts          0.534605


In [11]:
# 1. By borough — more likely to show real variation (funding/staffing/density differences)
critical_by_boro = df_clean[df_clean['CRITICAL FLAG'] == 'Critical'].groupby('BORO').size()
total_by_boro = df_clean.groupby('BORO').size()
print("Critical rate by borough:")
print((critical_by_boro / total_by_boro).sort_values(ascending=False))

# 2. Repeat offenders — restaurants with multiple critical violations (individual risk, not category risk)
repeat_offenders = (
    df_clean[df_clean['CRITICAL FLAG'] == 'Critical']
    .groupby(['CAMIS', 'DBA'])
    .size()
    .sort_values(ascending=False)
    .head(20)
)
print("\nTop repeat critical-violation restaurants:")
print(repeat_offenders)

# 3. Violation type breakdown — WHAT is actually going wrong most often
print("\nTop violation descriptions:")
print(df_clean['VIOLATION DESCRIPTION'].value_counts().head(15))

Critical rate by borough:
BORO
Staten Island    0.553979
Queens           0.532420
Brooklyn         0.528202
Bronx            0.521935
Manhattan        0.516110
Unknown          0.482940
dtype: float64

Top repeat critical-violation restaurants:
CAMIS     DBA                                 
50001215  BYUNG CHUN SOON DAE                     56
50138270  MEEM SPICY GROCERY AND DELI             56
50111296  BIG WONG                                51
50119363  DA PAI DON                              49
50078726  LINDA AZOGUENITA BAKERY & RESTAURANT    48
50139259  SHANGHAI TIME                           46
50040296  GAMMEEOK                                44
41406895  SUN SAI GAI RESTAURANT                  44
41468442  LAS PALMAS BAKERY                       43
41686215  SHUN WANG                               41
50106885  DON ALEX                                40
50056793  ASIAN KABAB & CURRY                     40
41696014  NOISETTE                                39
50120994  RUYI LAN

In [12]:
# Look at ONE repeat offender's history over time — does it show a worsening trend, or just noise?
example = df_clean[df_clean['DBA'] == 'BYUNG CHUN SOON DAE'].sort_values('INSPECTION DATE')
print(example[['INSPECTION DATE', 'CRITICAL FLAG', 'SCORE', 'VIOLATION DESCRIPTION']].to_string())

       INSPECTION DATE CRITICAL FLAG  SCORE                                                                                                                                                                                                                                                                                                                VIOLATION DESCRIPTION
268859      2024-03-14  Not Critical   47.0                                                                                                              Non-food contact surface or equipment made of unacceptable material, not kept clean, or not properly sealed, raised, spaced or movable to allow accessibility for cleaning on all sides, above and underneath the unit.
205607      2024-03-14      Critical   47.0                                                                                                                                                                                          Hot TCS food item that has been c

In [13]:
example = df_clean[df_clean['DBA'] == 'BYUNG CHUN SOON DAE'].sort_values('INSPECTION DATE')

print("Total inspections:", len(example))
print("Date range:", example['INSPECTION DATE'].min(), "to", example['INSPECTION DATE'].max())
print("Critical violations by year:")
print(example[example['CRITICAL FLAG']=='Critical']['INSPECTION DATE'].dt.year.value_counts().sort_index())

Total inspections: 85
Date range: 2024-03-14 00:00:00 to 2026-07-14 00:00:00
Critical violations by year:
INSPECTION DATE
2024    12
2025    34
2026    10
Name: count, dtype: int64


In [14]:
# For each top repeat offender, compare their most recent year's critical count vs their earliest year's
top_offenders = df_clean[df_clean['CRITICAL FLAG']=='Critical'].groupby('DBA').size().sort_values(ascending=False).head(10).index

for name in top_offenders:
    sub = df_clean[(df_clean['DBA']==name) & (df_clean['CRITICAL FLAG']=='Critical')]
    yearly = sub['INSPECTION DATE'].dt.year.value_counts().sort_index()
    trend = "rising" if yearly.iloc[-1] > yearly.iloc[0] else "falling/flat"
    print(name, "-", trend)

DUNKIN - rising
SUBWAY - rising
MCDONALD'S - rising
STARBUCKS - rising
POPEYES - rising
DUNKIN' - rising
KENNEDY FRIED CHICKEN - rising
GOLDEN KRUST CARIBBEAN BAKERY & GRILL - rising
PARIS BAGUETTE - rising
BURGER KING - rising


In [15]:
top_offenders = df_clean[df_clean['CRITICAL FLAG']=='Critical'].groupby('CAMIS').size().sort_values(ascending=False).head(10).index

for camis_id in top_offenders:
    sub = df_clean[(df_clean['CAMIS']==camis_id) & (df_clean['CRITICAL FLAG']=='Critical')]
    name = sub['DBA'].iloc[0]
    yearly = sub['INSPECTION DATE'].dt.year.value_counts().sort_index()
    trend = "rising" if yearly.iloc[-1] > yearly.iloc[0] else "falling/flat"
    print(name, camis_id, "-", trend, "-", dict(yearly))

BYUNG CHUN SOON DAE 50001215 - falling/flat - {2024: np.int64(12), 2025: np.int64(34), 2026: np.int64(10)}
MEEM SPICY GROCERY AND DELI 50138270 - falling/flat - {2023: np.int64(12), 2024: np.int64(13), 2025: np.int64(23), 2026: np.int64(8)}
BIG WONG 50111296 - rising - {2024: np.int64(9), 2025: np.int64(31), 2026: np.int64(11)}
DA PAI DON 50119363 - falling/flat - {2024: np.int64(11), 2025: np.int64(34), 2026: np.int64(4)}
LINDA AZOGUENITA BAKERY & RESTAURANT 50078726 - rising - {2023: np.int64(4), 2024: np.int64(3), 2025: np.int64(20), 2026: np.int64(21)}
SHANGHAI TIME 50139259 - falling/flat - {2024: np.int64(24), 2025: np.int64(14), 2026: np.int64(8)}
GAMMEEOK 50040296 - falling/flat - {2022: np.int64(6), 2023: np.int64(23), 2024: np.int64(11), 2025: np.int64(4)}
SUN SAI GAI RESTAURANT 41406895 - falling/flat - {2023: np.int64(22), 2025: np.int64(21), 2026: np.int64(1)}
LAS PALMAS BAKERY 41468442 - rising - {2023: np.int64(7), 2024: np.int64(8), 2025: np.int64(16), 2026: np.int64(12

In [16]:
citywide_yearly = df_clean[df_clean['CRITICAL FLAG']=='Critical']['INSPECTION DATE'].dt.year.value_counts().sort_index()
print(citywide_yearly)

INSPECTION DATE
2007       23
2008       15
2009        2
2010        4
2011        4
2012        3
2014        8
2015       14
2016      123
2017      240
2018      354
2019      418
2020      113
2021      193
2022     8813
2023    27169
2024    41747
2025    46949
2026    28759
Name: count, dtype: int64


In [17]:
df_recent = df_clean[df_clean['INSPECTION DATE'].dt.year >= 2022].copy()
print(df_recent.shape)

(288602, 28)


In [18]:
top_offenders = df_recent[df_recent['CRITICAL FLAG']=='Critical'].groupby('CAMIS').size().sort_values(ascending=False).head(10).index

for camis_id in top_offenders:
    sub = df_recent[(df_recent['CAMIS']==camis_id) & (df_recent['CRITICAL FLAG']=='Critical')]
    name = sub['DBA'].iloc[0]
    yearly = sub['INSPECTION DATE'].dt.year.value_counts().sort_index()
    trend = "rising" if yearly.iloc[-1] > yearly.iloc[0] else "falling/flat"
    print(name, "-", trend, "-", dict(yearly))

BYUNG CHUN SOON DAE - falling/flat - {2024: np.int64(12), 2025: np.int64(34), 2026: np.int64(10)}
MEEM SPICY GROCERY AND DELI - falling/flat - {2023: np.int64(12), 2024: np.int64(13), 2025: np.int64(23), 2026: np.int64(8)}
BIG WONG - rising - {2024: np.int64(9), 2025: np.int64(31), 2026: np.int64(11)}
DA PAI DON - falling/flat - {2024: np.int64(11), 2025: np.int64(34), 2026: np.int64(4)}
LINDA AZOGUENITA BAKERY & RESTAURANT - rising - {2023: np.int64(4), 2024: np.int64(3), 2025: np.int64(20), 2026: np.int64(21)}
SHANGHAI TIME - falling/flat - {2024: np.int64(24), 2025: np.int64(14), 2026: np.int64(8)}
GAMMEEOK - falling/flat - {2022: np.int64(6), 2023: np.int64(23), 2024: np.int64(11), 2025: np.int64(4)}
SUN SAI GAI RESTAURANT - falling/flat - {2023: np.int64(22), 2025: np.int64(21), 2026: np.int64(1)}
LAS PALMAS BAKERY - rising - {2023: np.int64(7), 2024: np.int64(8), 2025: np.int64(16), 2026: np.int64(12)}
SHUN WANG - falling/flat - {2024: np.int64(13), 2025: np.int64(26), 2026: np.i

In [19]:
compare = df_recent[df_recent['CRITICAL FLAG']=='Critical']
compare = compare[compare['INSPECTION DATE'].dt.year.isin([2024,2025])]
change = compare.groupby([compare['INSPECTION DATE'].dt.year, 'VIOLATION DESCRIPTION']).size().unstack(level=0)
change['pct_change'] = ((change[2025] - change[2024]) / change[2024] * 100)
print(change.sort_values('pct_change', ascending=False).head(10)[[2024,2025,'pct_change']])

INSPECTION DATE                                      2024   2025  pct_change
VIOLATION DESCRIPTION                                                       
Food containing a prohibited substance held, ke...   33.0   75.0  127.272727
Food worker or vendor working or is knowingly o...    1.0    2.0  100.000000
Food contact surface, refillable, reusable cont...  106.0  207.0   95.283019
Time/Temperature Control for Safety (TCS) food ...   50.0   89.0   78.000000
Juice packaged on premises with no or incomplet...   74.0  116.0   56.756757
Meat, fish, molluscan shellfish, unpasteurized ...   48.0   74.0   54.166667
Food, prohibited, from unapproved or unknown so...  292.0  447.0   53.082192
Unclean or cracked whole eggs or unpasteurized ...   21.0   32.0   52.380952
Toilet facility not provided for employees or f...   50.0   71.0   42.000000
No approved written standard operating procedur...  239.0  329.0   37.656904


In [20]:
change['abs_change'] = change[2025] - change[2024]
print(change.sort_values('abs_change', ascending=False).head(10)[[2024,2025,'abs_change']])

INSPECTION DATE                                       2024    2025  abs_change
VIOLATION DESCRIPTION                                                         
Food, supplies, or equipment not protected from...  4721.0  5896.0      1175.0
Cold TCS food item held above 41 °F; smoked or ...  5045.0  5850.0       805.0
Hot TCS food item not held at or above 140 °F.      4117.0  4857.0       740.0
Food Protection Certificate (FPC) not held by m...  2043.0  2475.0       432.0
No hand washing facility in or adjacent to toil...  1397.0  1785.0       388.0
Evidence of mice or live mice in establishment'...  3797.0  4179.0       382.0
Food contact surface not properly washed, rinse...  5050.0  5426.0       376.0
Sanitized equipment or utensil, including in-us...  1299.0  1652.0       353.0
Evidence of rats or live rats in establishment'...   745.0   910.0       165.0
Properly scaled and calibrated thermometer or t...   515.0   673.0       158.0


In [21]:
df_recent.to_csv("restaurant_inspections_processed.csv", index=False)